In [0]:
-- ============================================================
-- CineClub Diwali Offer (LOY-00723) — canvas b73d9b15-83a1-4929-8565-04c3ed0b48e3
-- Offer window: 2026-10-26 00:00 to 2026-11-09 00:00 ET
-- ============================================================

-- Cell 1: openers (one row per cde_id), first opened in the offer window
CREATE OR REPLACE TEMP VIEW openers AS
WITH opens AS (
  SELECT user_id AS braze_id,
         MAX(external_user_id) AS cc_user_profile_guid,
         MIN(from_unixtime(time)) AS first_open_ts,
         COUNT(*) AS open_count
  FROM cpx_dataanalytics_gold.marketing.mkt_braze_users_messages_email_open
  WHERE canvas_id = 'b73d9b15-83a1-4929-8565-04c3ed0b48e3'
  GROUP BY user_id
),
cust AS (
  SELECT DISTINCT cc_user_profile_guid, cde_id
  FROM cpx_dataanalytics_gold.customer360.cpx_d_customer
  WHERE cc_user_profile_guid IS NOT NULL
)
SELECT c.cde_id,
       from_utc_timestamp(MIN(o.first_open_ts), 'America/New_York') AS first_open_et,
       SUM(o.open_count) AS open_count
FROM opens o
JOIN cust c ON c.cc_user_profile_guid = o.cc_user_profile_guid
GROUP BY c.cde_id
HAVING first_open_et < TIMESTAMP '2026-11-09 00:00:00';

-- Cell 2: openers' CineClub profile IDs (events and gifting are keyed on cc_user_profile_id)
CREATE OR REPLACE TEMP VIEW opener_ids AS
SELECT DISTINCT o.cde_id, d.cc_user_profile_id
FROM openers o
JOIN cpx_dataanalytics_gold.customer360.cpx_d_customer d ON d.cde_id = o.cde_id
WHERE d.cc_user_profile_id IS NOT NULL;

-- Cell 3: qualifying actions in the window (all members; filtered to openers later)
CREATE OR REPLACE TEMP VIEW offer_actions AS
WITH created_all AS (                                   -- KPI-02/03 need full history
  SELECT cc_user_profile_id, plan_id, changed,
         from_utc_timestamp(reporting_datetime, 'America/New_York') AS action_et,
         COUNT(*) OVER (PARTITION BY cc_user_profile_id) AS n_created_ever
  FROM marketing.mkt_subscription_events
  WHERE event_type = 'subscription_created'
),
signups AS (
  SELECT cc_user_profile_id,
         CASE WHEN n_created_ever = 1 THEN '1. Net new sign-up' ELSE '2. Returning sign-up' END AS bucket,
         MIN(action_et) AS action_et, 1 AS n_items
  FROM created_all
  WHERE plan_id = 'annual1' AND COALESCE(changed, FALSE) = FALSE
    AND action_et >= TIMESTAMP '2026-10-26 00:00:00' AND action_et < TIMESTAMP '2026-11-09 00:00:00'
  GROUP BY 1, 2
),
upgrades AS (                                           -- BR-056; record_occurred_at = epoch seconds (UTC)
  SELECT cc_user_profile_id, '3. Upgrade' AS bucket,
         MIN(from_utc_timestamp(timestamp_seconds(record_occurred_at), 'America/New_York')) AS action_et, 1 AS n_items
  FROM marketing.mkt_subscription_events
  WHERE event_type = 'subscription_changed' AND changed = TRUE
    AND changed_from <> changed_to AND changed_to = 'annual1'
    AND from_utc_timestamp(timestamp_seconds(record_occurred_at), 'America/New_York') >= TIMESTAMP '2026-10-26 00:00:00'
    AND from_utc_timestamp(timestamp_seconds(record_occurred_at), 'America/New_York') <  TIMESTAMP '2026-11-09 00:00:00'
  GROUP BY 1
),
gifts AS (                                              -- BR-032; purchaser gets the points
  SELECT purchaser_id AS cc_user_profile_id, '4. Gift purchase' AS bucket,
         MIN(from_utc_timestamp(purchase_date, 'America/New_York')) AS action_et, COUNT(*) AS n_items
  FROM cpx_dataanalytics_silver.cineclub.gifting
  WHERE from_utc_timestamp(purchase_date, 'America/New_York') >= TIMESTAMP '2026-10-26 00:00:00'
    AND from_utc_timestamp(purchase_date, 'America/New_York') <  TIMESTAMP '2026-11-09 00:00:00'
  GROUP BY 1
)
SELECT * FROM signups
UNION ALL SELECT * FROM upgrades
UNION ALL SELECT * FROM gifts;

-- Cell 4: opener-level actions + 30-day cancellation flag (KPI-04 exits only)
CREATE OR REPLACE TEMP VIEW opener_actions AS
WITH cancels AS (
  SELECT cc_user_profile_id,
         from_utc_timestamp(reporting_datetime, 'America/New_York') AS cancel_et
  FROM marketing.mkt_subscription_events
  WHERE event_type = 'subscription_cancelled' AND changed IS NOT TRUE
    AND reporting_datetime >= TIMESTAMP '2026-10-26 00:00:00'
)
SELECT oi.cde_id, o.first_open_et, a.bucket,
       MIN(a.action_et) AS action_et,
       SUM(a.n_items)   AS n_items,
       MAX(CASE WHEN a.bucket <> '4. Gift purchase'
                 AND c.cancel_et >= a.action_et
                 AND c.cancel_et <  a.action_et + INTERVAL 30 DAYS THEN 1 ELSE 0 END) AS cancelled_30d
FROM offer_actions a
JOIN opener_ids oi ON oi.cc_user_profile_id = a.cc_user_profile_id
JOIN openers o     ON o.cde_id = oi.cde_id
LEFT JOIN cancels c ON c.cc_user_profile_id = a.cc_user_profile_id
GROUP BY oi.cde_id, o.first_open_et, a.bucket;

-- Cell 5: email funnel (member level, all recipients)
CREATE OR REPLACE TEMP VIEW email_funnel AS
WITH braze_map AS (                                     -- Braze user -> cde_id
  SELECT DISTINCT b.user_id AS braze_id, c.cde_id
  FROM (SELECT user_id, MAX(external_user_id) AS guid
        FROM cpx_dataanalytics_gold.marketing.mkt_braze_users_messages_email_send
        WHERE canvas_id = 'b73d9b15-83a1-4929-8565-04c3ed0b48e3'
        GROUP BY user_id) b
  JOIN (SELECT DISTINCT cc_user_profile_guid, cde_id
        FROM cpx_dataanalytics_gold.customer360.cpx_d_customer
        WHERE cc_user_profile_guid IS NOT NULL) c
    ON c.cc_user_profile_guid = b.guid
),
evt AS (
  SELECT 'sent' AS stage, user_id, time FROM cpx_dataanalytics_gold.marketing.mkt_braze_users_messages_email_send
    WHERE canvas_id = 'b73d9b15-83a1-4929-8565-04c3ed0b48e3'
  UNION ALL
  SELECT 'delivered', user_id, time FROM cpx_dataanalytics_gold.marketing.mkt_braze_users_messages_email_delivery
    WHERE canvas_id = 'b73d9b15-83a1-4929-8565-04c3ed0b48e3'
  UNION ALL
  SELECT 'opened', user_id, time FROM cpx_dataanalytics_gold.marketing.mkt_braze_users_messages_email_open
    WHERE canvas_id = 'b73d9b15-83a1-4929-8565-04c3ed0b48e3'
  UNION ALL
  SELECT 'clicked', user_id, time FROM cpx_dataanalytics_gold.marketing.mkt_braze_users_messages_email_click
    WHERE canvas_id = 'b73d9b15-83a1-4929-8565-04c3ed0b48e3'
),
member_stage AS (                                       -- one row per cde_id, first time per stage (ET)
  SELECT m.cde_id,
         MIN(CASE WHEN e.stage = 'sent'      THEN from_utc_timestamp(from_unixtime(e.time), 'America/New_York') END) AS sent_et,
         MIN(CASE WHEN e.stage = 'delivered' THEN from_utc_timestamp(from_unixtime(e.time), 'America/New_York') END) AS delivered_et,
         MIN(CASE WHEN e.stage = 'opened'    THEN from_utc_timestamp(from_unixtime(e.time), 'America/New_York') END) AS opened_et,
         MIN(CASE WHEN e.stage = 'clicked'   THEN from_utc_timestamp(from_unixtime(e.time), 'America/New_York') END) AS clicked_et
  FROM evt e
  JOIN braze_map m ON m.braze_id = e.user_id
  GROUP BY m.cde_id
),
cancels AS (                                            -- KPI-04 exits only
  SELECT cc_user_profile_id,
         from_utc_timestamp(reporting_datetime, 'America/New_York') AS cancel_et
  FROM marketing.mkt_subscription_events
  WHERE event_type = 'subscription_cancelled' AND changed IS NOT TRUE
    AND reporting_datetime >= TIMESTAMP '2026-10-26 00:00:00'
),
member_actions AS (                                     -- all actions, with 30-day cancellation flag
  SELECT d.cde_id, a.bucket, MIN(a.action_et) AS action_et, SUM(a.n_items) AS n_items,
         MAX(CASE WHEN a.bucket <> '4. Gift purchase'
                   AND c.cancel_et >= a.action_et
                   AND c.cancel_et <  a.action_et + INTERVAL 30 DAYS THEN 1 ELSE 0 END) AS cancelled_30d
  FROM offer_actions a
  JOIN (SELECT DISTINCT cc_user_profile_id, cde_id
        FROM cpx_dataanalytics_gold.customer360.cpx_d_customer
        WHERE cc_user_profile_id IS NOT NULL) d
    ON d.cc_user_profile_id = a.cc_user_profile_id
  LEFT JOIN cancels c
    ON c.cc_user_profile_id = a.cc_user_profile_id
  GROUP BY d.cde_id, a.bucket
)
SELECT s.cde_id, s.sent_et, s.delivered_et, s.opened_et, s.clicked_et,
       ma.bucket, ma.action_et, ma.n_items, ma.cancelled_30d,
       CASE WHEN s.clicked_et IS NOT NULL AND ma.action_et >= s.clicked_et THEN 1 ELSE 0 END AS converted_after_click
FROM member_stage s
LEFT JOIN member_actions ma ON ma.cde_id = s.cde_id;

-- ============================================================
-- OUTPUT VIEWS
-- ============================================================

-- Output 3 (built first; outputs 1 and 2 read from it): one row per converter × bucket, with timing
CREATE OR REPLACE TEMP VIEW out_cde_list AS
SELECT cde_id,
       bucket,
       n_items,                                         -- gifts = # gifts bought
       cancelled_30d,
       action_et,
       delivered_et,
       opened_et,
       clicked_et,
       CASE WHEN action_et < delivered_et                                       THEN 'Convert Before Receiving'
            WHEN clicked_et < TIMESTAMP '2026-11-09' AND action_et >= clicked_et THEN 'Convert After Clicking'
            WHEN opened_et  < TIMESTAMP '2026-11-09' AND action_et >= opened_et  THEN 'Convert After Opening'
            WHEN opened_et  < TIMESTAMP '2026-11-09'                             THEN 'Convert Before Opening'
            ELSE                                                                      'Convert Without Opening'
       END AS conversion_timing
FROM email_funnel
WHERE bucket IS NOT NULL
  AND delivered_et IS NOT NULL;

-- Output 1: funnel totals
CREATE OR REPLACE TEMP VIEW out_funnel AS
WITH m AS (
  SELECT cde_id,
         MAX(CASE WHEN delivered_et IS NOT NULL THEN 1 ELSE 0 END)                                AS is_received,
         MAX(CASE WHEN delivered_et IS NOT NULL AND opened_et  < TIMESTAMP '2026-11-09' THEN 1 ELSE 0 END) AS is_opened,
         MAX(CASE WHEN delivered_et IS NOT NULL AND clicked_et < TIMESTAMP '2026-11-09' THEN 1 ELSE 0 END) AS is_clicked
  FROM email_funnel
  GROUP BY cde_id
),
f AS (
  SELECT SUM(is_received) AS received, SUM(is_opened) AS opened, SUM(is_clicked) AS clicked,
         (SELECT COUNT(DISTINCT cde_id) FROM out_cde_list) AS converted
  FROM m
)
SELECT '1. Received'  AS stage, received  AS members, 100.00                               AS pct_of_received FROM f
UNION ALL SELECT '2. Opened',    opened,    ROUND(100 * opened    / received, 2) FROM f
UNION ALL SELECT '3. Clicked',   clicked,   ROUND(100 * clicked   / received, 2) FROM f
UNION ALL SELECT '4. Converted', converted, ROUND(100 * converted / received, 2) FROM f;

-- Output 2: conversions by bucket
CREATE OR REPLACE TEMP VIEW out_buckets AS
SELECT CASE bucket
         WHEN '1. Net new sign-up'   THEN '1. Net New'
         WHEN '2. Returning sign-up' THEN '2. Returning'
         WHEN '3. Upgrade'           THEN '3. Upgraded'
         WHEN '4. Gift purchase'     THEN '4. Gift Purchasers'
       END                       AS bucket,
       COUNT(DISTINCT cde_id)    AS members,
       SUM(n_items)              AS actions                -- gifts = # gifts bought
FROM out_cde_list
GROUP BY 1
ORDER BY 1;

-- ============================================================
-- RESULTS (in a %sql notebook cell only the last one displays — run each in its own cell if needed)
-- ============================================================
SELECT * FROM out_funnel   ORDER BY stage;
SELECT * FROM out_buckets  ORDER BY bucket;
SELECT * FROM out_cde_list ORDER BY conversion_timing, bucket, action_et;